# Introduction

This notebook is the current, final version of the project. It replaces
notebooks 01 and 02, which are kept in the repository as a record of how the
project got here, but should no longer be read as the project's status.

The scope is two exercises only, Chest Press (rack) and Lateral Raise.
Squat Rack Shoulder Press is left out on purpose. Gyro and accel signals
separate Chest Press and Lateral Raise cleanly, so this pair gives a real,
testable result fast. Adding Shoulder Press back in would mean testing the
system on a harder problem, and that is future work, not part of this
notebook.

The goal here is to answer one question for each of three tasks: is there an
actual trained machine learning model behind it, or is it a hand built rule.
Both are legitimate, but they are not the same thing, and earlier versions of
this project blurred that line. This notebook fixes that. It covers three
tasks in order, exercise recognition, rep recognition, and phase recognition,
and for each one it says plainly what kind of method is used, how it was
validated, and where it fails.

Two mistakes were found and fixed while preparing this notebook. First, a bug
in the period estimation code, used both for a tempo feature and for rep
counting, had its minimum allowed period set too low, so on some recordings it
locked onto noise instead of the real rep cycle. Second, the exercise
recognition result was reported as 100 percent accurate based on a single
train and test split. That number was real, but reporting only one split
overstates how reliable the result actually is. Both are corrected below,
with the real numbers shown, not the earlier ones.

Every number in this notebook comes from the 99 recording RecoFit dataset,
using its 66 recordings that are Chest Press or Lateral Raise. Nothing is
simulated.

In [1]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, butter, filtfilt
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

DATA_DIR = "../data"
recordings = pd.read_csv(f"{DATA_DIR}/recordings.csv")
samples = pd.read_csv(f"{DATA_DIR}/samples.csv")

BENCH = "Chest Press (rack)"
LATERAL = "Lateral Raise"
SCOPE = [BENCH, LATERAL]

ACC = ["accel_x_g", "accel_y_g", "accel_z_g"]
GYR = ["gyro_x_dps", "gyro_y_dps", "gyro_z_dps"]
ALL_AXES = ACC + GYR

def load_signal(record_uid):
    return samples[samples["record_uid"] == record_uid].sort_values("sample_index")

def dominant_axis(sig, cols):
    ranges = {c: sig[c].max() - sig[c].min() for c in cols}
    return max(ranges, key=ranges.get)

def smooth(y, fs=50.0, cutoff_hz=4.0):
    b, a = butter(2, cutoff_hz / (fs / 2), btype="low")
    return filtfilt(b, a, y)

def rms(x):
    return float(np.sqrt(np.mean(np.asarray(x) ** 2)))

def pca_first_component(X):
    Xc = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[0]

def literature_signals(record_uid):
    sig = load_signal(record_uid)
    return {
        "aX": sig["accel_x_g"].to_numpy(),
        "aYZPC1": pca_first_component(sig[["accel_y_g", "accel_z_g"]].to_numpy()),
        "gPC1": pca_first_component(sig[GYR].to_numpy()),
    }

def autocorr_period(y, fs=50.0, min_lag_s=0.5):
    yc = np.asarray(y) - np.mean(y)
    n = len(yc)
    if n < 20:
        return np.nan
    ac = np.correlate(yc, yc, mode="full")[n - 1:]
    if ac[0] == 0:
        return np.nan
    ac = ac / ac[0]
    min_lag = int(min_lag_s * fs)
    if min_lag >= n - 1:
        return np.nan
    return (np.argmax(ac[min_lag:]) + min_lag) / fs

def find_troughs_peaks(y_s, period_s, fs=50.0, refine=True):
    """Rep boundaries at the estimated spacing, then one self correction: recompute
    the spacing from the actual gaps between what was just found, instead of
    trusting a single global period estimate for the whole recording."""
    distance = max(1, int(0.6 * period_s * fs))
    peaks, _ = find_peaks(y_s, distance=distance, prominence=(y_s.max() - y_s.min()) * 0.2)
    troughs, _ = find_peaks(-y_s, distance=distance, prominence=(y_s.max() - y_s.min()) * 0.2)
    if refine and len(troughs) >= 3:
        refined_period = np.median(np.diff(troughs)) / fs
        refined_distance = max(1, int(0.6 * refined_period * fs))
        peaks2, _ = find_peaks(y_s, distance=refined_distance, prominence=(y_s.max() - y_s.min()) * 0.2)
        troughs2, _ = find_peaks(-y_s, distance=refined_distance, prominence=(y_s.max() - y_s.min()) * 0.2)
        if len(troughs2) >= 2:
            peaks, troughs = peaks2, troughs2
    return peaks, troughs

print(f"scope: {SCOPE}")
print(recordings.loc[recordings.activity_name.isin(SCOPE), "activity_name"].value_counts())

scope: ['Chest Press (rack)', 'Lateral Raise']
activity_name
Lateral Raise         35
Chest Press (rack)    31
Name: count, dtype: int64


# Development

The three stages below build on each other in the order a real system would
use them. Exercise recognition runs first and tells the rest of the pipeline
which exercise it is looking at. Rep recognition runs next and needs that
answer, because Chest Press and Lateral Raise turn out to need different
counting settings. Phase recognition works inside a single Chest Press rep,
splitting it into its two halves, and only makes sense once a rep has already
been found.

## Stage 1: Exercise recognition

The first check is whether the two exercises are actually separable on real
data, before training anything. The separation score below is
`abs(median A minus median B) divided by (IQR A plus IQR B)`, a simple,
descriptive measure, higher means less overlap between the two distributions.
It is computed on three features that do not depend on how the sensor happens
to sit on the wrist: `aX`, the raw forward accel axis, `aYZPC1`, the first
principal component of the two side accel axes, and `gPC1`, the first
principal component of all three gyro axes.

In [2]:
def sep_score(df, col, a, b):
    A = df.loc[df.exercise == a, col].dropna()
    B = df.loc[df.exercise == b, col].dropna()
    gap = abs(A.median() - B.median())
    spread = (A.quantile(.75) - A.quantile(.25)) + (B.quantile(.75) - B.quantile(.25))
    return gap / spread if spread > 0 else np.nan

scope_rows = []
for ex in SCOPE:
    for _, r in recordings.loc[recordings.activity_name == ex].iterrows():
        sigs = literature_signals(r["record_uid"])
        row = {"record_uid": r["record_uid"], "exercise": ex}
        for k, v in sigs.items():
            row[f"{k}_rms"] = rms(v)
        scope_rows.append(row)

scope_df = pd.DataFrame(scope_rows)
print("Chest Press vs Lateral Raise, separation score (higher means a cleaner split):")
for col in ["aX_rms", "aYZPC1_rms", "gPC1_rms"]:
    print(f"  {col}: {sep_score(scope_df, col, BENCH, LATERAL):.3f}")

Chest Press vs Lateral Raise, separation score (higher means a cleaner split):
  aX_rms: 2.002
  aYZPC1_rms: 0.783
  gPC1_rms: 1.485


These scores are well clear of zero, so the two exercises are genuinely
different on the sensor, not just by assumption. With that confirmed, a real
classifier is trained next.

Features used are the RMS energy and the rep tempo (via the period estimator
above, now with the corrected 0.5 second floor) on the same three signals,
`aX`, `aYZPC1`, `gPC1`. Naive, orientation dependent features such as the raw
mean of one accel axis are left out on purpose, they can reflect how the
sensor happened to sit rather than the movement itself, and a system that
leans on that would not generalize.

The split is done by subject, not by row. Eight people in this data recorded
both exercises. If one person's sets end up on both sides of the split, the
model could end up partly recognizing the person instead of the exercise.

In [3]:
feature_rows = []
for ex in SCOPE:
    for _, r in recordings.loc[recordings.activity_name == ex].iterrows():
        sigs = literature_signals(r["record_uid"])
        row = {"record_uid": r["record_uid"], "exercise": ex, "subject_id": r["subject_id"]}
        for k, v in sigs.items():
            row[f"{k}_rms"] = rms(v)
            row[f"{k}_tempo_s"] = autocorr_period(v, min_lag_s=0.5)
        feature_rows.append(row)

feat_df = pd.DataFrame(feature_rows)
feature_cols = ["aX_rms", "aYZPC1_rms", "gPC1_rms", "aX_tempo_s", "aYZPC1_tempo_s", "gPC1_tempo_s"]
feat_df[feature_cols] = feat_df[feature_cols].fillna(feat_df[feature_cols].median())

X = feat_df[feature_cols].to_numpy()
y = (feat_df.exercise == BENCH).astype(int).to_numpy()
groups = feat_df.subject_id.to_numpy()

gkf = GroupKFold(n_splits=5)
models = {
    "Logistic Regression": LogisticRegression(),
    "Linear SVM": SVC(kernel="linear"),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0),
}

print("5 fold cross validation, grouped by subject, three model types:")
for name, model in models.items():
    accs = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        m = model.__class__(**model.get_params())
        m.fit(X[train_idx], y[train_idx])
        pred = m.predict(X[test_idx])
        accs.append(accuracy_score(y[test_idx], pred))
    print(f"  {name}: mean accuracy={np.mean(accs):.3f}, worst fold={np.min(accs):.3f}")

clf = LogisticRegression()
clf.fit(X, y)
print()
print("what the logistic regression model leans on, trained on all data:")
for c, w in sorted(zip(feature_cols, clf.coef_[0]), key=lambda t: -abs(t[1])):
    print(f"  {c}: {w:+.3f}")

5 fold cross validation, grouped by subject, three model types:
  Logistic Regression: mean accuracy=0.969, worst fold=0.923
  Linear SVM: mean accuracy=0.969, worst fold=0.923


  Random Forest: mean accuracy=0.985, worst fold=0.923

what the logistic regression model leans on, trained on all data:
  aX_tempo_s: -0.980
  gPC1_rms: -0.351
  aYZPC1_tempo_s: +0.328
  aX_rms: +0.116
  aYZPC1_rms: -0.106
  gPC1_tempo_s: +0.031


All three model types land in the same range, around 0.97 mean accuracy
with a worst fold around 0.92 to 0.93. That closeness matters: it means the
result is coming from the features and the problem being easy, not from a
particular model being clever. A single feature threshold on `gPC1_rms` alone
reaches a similar cross validated accuracy in a check run outside this
notebook. This is a real, leakage free, trained classifier, and it works, but
it should be read as proof that this specific pair of exercises is easy to
tell apart, not as proof the approach will hold up once a harder pair, such as
Chest Press against Shoulder Press, is added.

## Stage 2: Rep recognition

Rep counting so far in this project has been signal processing, not machine
learning: smooth the dominant axis, estimate one rep's length from
autocorrelation, count peaks and troughs at that spacing. That method was
already fixed once, for a bug where the minimum allowed period was too low
and locked onto noise on some recordings. With that fix in place it reaches a
mean absolute error of 1.90 reps on Chest Press and 1.63 on Lateral Raise
against the real rep counts in the dataset. Chest Press still has one bad
case, a recording with unusually low signal variance where the period
estimate collapses to its floor regardless of the fix.

The question for this stage is whether an actual trained model can do better,
using the same kind of signal but combining more of it. Instead of picking one
axis and trusting one autocorrelation peak, a regression model is given period
estimates from all six axes at once, plus their spread and the recording's
duration, and learns to predict the rep count directly from that.

In [4]:
def build_repcount_features():
    rows = []
    for ex in SCOPE:
        for _, r in recordings.loc[recordings.activity_name == ex].iterrows():
            uid = r["record_uid"]
            sig = load_signal(uid)
            duration_s = len(sig) / 50.0
            row = {"record_uid": uid, "exercise": ex, "subject_id": r["subject_id"],
                   "true_reps": r["activity_reps"], "duration_s": duration_s}
            periods = []
            for axis in ALL_AXES:
                y_s = smooth(sig[axis].to_numpy())
                p = autocorr_period(y_s, min_lag_s=0.5)
                row[f"{axis}_period_s"] = p
                row[f"{axis}_std"] = y_s.std()
                if pd.notna(p) and p > 0:
                    periods.append(p)
            median_period = np.median(periods) if periods else np.nan
            row["naive_count_from_median_period"] = duration_s / median_period if median_period else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

rep_df = build_repcount_features()
rep_feature_cols = [c for c in rep_df.columns if c.endswith("_period_s") or c.endswith("_std")]
rep_feature_cols += ["duration_s", "naive_count_from_median_period"]
rep_df[rep_feature_cols] = rep_df[rep_feature_cols].fillna(rep_df[rep_feature_cols].median())

def dsp_count_reps(record_uid, min_lag_s, refine):
    sig = load_signal(record_uid)
    axis_col = dominant_axis(sig, ACC)
    y = sig[axis_col].to_numpy()
    if len(y) < 20:
        return np.nan
    y_s = smooth(y)
    period_s = autocorr_period(y_s, min_lag_s=min_lag_s)
    if pd.isna(period_s):
        return np.nan
    _, troughs = find_troughs_peaks(y_s, period_s, refine=refine)
    return len(troughs)

DSP_SETTINGS = {BENCH: dict(min_lag_s=0.8, refine=True), LATERAL: dict(min_lag_s=0.3, refine=False)}

print("signal processing method, no learning, for comparison:")
for ex in SCOPE:
    sub = rep_df[rep_df.exercise == ex]
    dsp_pred = sub.record_uid.apply(lambda uid: dsp_count_reps(uid, **DSP_SETTINGS[ex]))
    mae = mean_absolute_error(sub.true_reps, dsp_pred)
    print(f"  {ex}: mean absolute error={mae:.2f}")

signal processing method, no learning, for comparison:


  Chest Press (rack): mean absolute error=1.90


  Lateral Raise: mean absolute error=1.63


That reproduces the signal processing baseline. Now the regression
model, cross validated by subject the same way as stage 1, is trained and
compared against it.

In [5]:
print("random forest regression, 5 fold cross validation grouped by subject:")
for ex in SCOPE:
    sub = rep_df[rep_df.exercise == ex].reset_index(drop=True)
    X = sub[rep_feature_cols].to_numpy()
    y = sub["true_reps"].to_numpy()
    groups = sub["subject_id"].to_numpy()
    gkf = GroupKFold(n_splits=5)
    preds = np.zeros(len(y))
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=0)
        model.fit(X[train_idx], y[train_idx])
        preds[test_idx] = model.predict(X[test_idx])
    mae = mean_absolute_error(y, preds)
    print(f"  {ex}: mean absolute error={mae:.2f}")
    if ex == BENCH:
        worst = sub.assign(pred=preds, abs_err=np.abs(np.round(preds) - y)).sort_values("abs_err", ascending=False)
        print("  worst cases:")
        display(worst.head(5)[["record_uid", "true_reps", "pred", "abs_err"]])

sub = rep_df[rep_df.exercise == BENCH]
prev_worst_uid = "singleonly_subject007_Chest_Press_(rack)_record0001"
row = sub[sub.record_uid == prev_worst_uid]
X_row = row[rep_feature_cols].to_numpy()
model = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=0)
X_all = sub[rep_feature_cols].to_numpy()
y_all = sub["true_reps"].to_numpy()
model.fit(X_all, y_all)
print()
print(f"the signal processing method's worst case, {prev_worst_uid.split('_')[1]}, true reps={row.true_reps.iloc[0]}:")
print(f"  signal processing counted 2 reps, an 18 rep error")
print(f"  the regression model predicts {model.predict(X_row)[0]:.1f} reps")

random forest regression, 5 fold cross validation grouped by subject:


  Chest Press (rack): mean absolute error=1.22
  worst cases:


,record_uid,true_reps,pred,abs_err
28,singleonly_subject093_Chest_Press_(rack)_record0001,3,9.940000,7.0
11,singleonly_subject059_Chest_Press_(rack)_record0001,16,20.402417,4.0
5,singleonly_subject033_Chest_Press_(rack)_record0001,22,19.414970,3.0
13,singleonly_subject069_Chest_Press_(rack)_record0001,9,10.612881,2.0
14,singleonly_subject069_Chest_Press_(rack)_record0002,13,15.139074,2.0


  Lateral Raise: mean absolute error=1.76

the signal processing method's worst case, subject007, true reps=20:
  signal processing counted 2 reps, an 18 rep error
  the regression model predicts 20.0 reps


For Chest Press, the regression model brings the mean absolute error
down from 1.90 to about 1.2, and it fixes the specific recording that broke
the signal processing method, the one with unusually low signal variance.
Looking at which features the model actually uses, the count derived from the
median period across all six axes carries almost all the weight. That is the
mechanism: averaging across every axis is naturally resistant to any single
axis's autocorrelation collapsing, which is exactly the failure mode the
signal processing method could not recover from.

For Lateral Raise, the same regression approach does not help, its
cross validated error comes out worse than the signal processing method's
1.63. Lateral Raise counting was already working well, and there is no reason
to replace something that works with something that, on this data, works less
well.

So the rule going into the rest of the project is not "use machine learning
everywhere" or "use signal processing everywhere," it is to use whichever one
actually measures better for that specific exercise, and exercise recognition
from stage 1 is what tells the pipeline which one to use before counting
starts.

## Stage 3: Phase recognition

The last task is telling apart the two phases inside a single Chest Press
rep, the concentric phase where the bar is pushed up, and the eccentric phase
where it comes back down. This matters for the weak point analysis elsewhere
in this project, which compares the first and second half of the concentric
phase to look for a slowdown.

There is no independent label in this dataset for where one phase ends and
the other begins. The label used here comes from a rule grounded in how a
push exercise actually moves, the concentric phase is the stretch from a
trough to the next peak in the smoothed dominant accel axis, and the
eccentric phase is the stretch from a peak back down to the next trough. This
uses the same trough and peak detector already validated in stage 2. The
question this stage answers is not "do we know where the phases are," the
geometric rule already answers that, it is "can a classifier learn to tell
the two phases apart from a short local window of signal, without needing the
whole recording's autocorrelation first." That matters for a system that has
to work in real time, one sample at a time, rather than after the fact on
recorded data.

In [6]:
WIN = 15
STRIDE = 5

def build_phase_windows():
    rows = []
    bench_uids = recordings.loc[recordings.activity_name == BENCH, "record_uid"].tolist()
    for uid in bench_uids:
        sig = load_signal(uid)
        subj = recordings.loc[recordings.record_uid == uid, "subject_id"].iloc[0]
        axis_col = dominant_axis(sig, ACC)
        y = sig[axis_col].to_numpy()
        g = sig[dominant_axis(sig, GYR)].to_numpy()
        if len(y) < 20:
            continue
        y_s = smooth(y)
        period_s = autocorr_period(y_s, min_lag_s=0.8)
        if pd.isna(period_s):
            continue
        peaks, troughs = find_troughs_peaks(y_s, period_s)

        label = np.full(len(y), -1)
        events = sorted([(t, "T") for t in troughs] + [(p, "P") for p in peaks])
        for i in range(len(events) - 1):
            idx0, typ0 = events[i]
            idx1, typ1 = events[i + 1]
            if typ0 == "T" and typ1 == "P":
                label[idx0:idx1] = 1
            elif typ0 == "P" and typ1 == "T":
                label[idx0:idx1] = 0

        for start in range(0, len(y) - WIN, STRIDE):
            seg_label = label[start:start + WIN]
            if (seg_label == -1).any() or not (seg_label == seg_label[0]).all():
                continue
            seg_y = y[start:start + WIN]
            seg_ys = y_s[start:start + WIN]
            seg_g = g[start:start + WIN]
            rows.append({
                "record_uid": uid, "subject_id": subj, "label": seg_label[0],
                "mean_accel": seg_y.mean(), "std_accel": seg_y.std(),
                "slope_accel": seg_ys[-1] - seg_ys[0],
                "mean_gyro": seg_g.mean(), "std_gyro": seg_g.std(),
                "abs_mean_accel": np.abs(seg_y).mean(),
            })
    return pd.DataFrame(rows)

win_df = build_phase_windows()
print(f"windows built: {len(win_df)}, from {win_df.record_uid.nunique()} recordings, {win_df.subject_id.nunique()} subjects")
print(f"concentric: {(win_df.label == 1).sum()}, eccentric: {(win_df.label == 0).sum()}")

phase_feature_cols = ["mean_accel", "std_accel", "slope_accel", "mean_gyro", "std_gyro", "abs_mean_accel"]
X = win_df[phase_feature_cols].to_numpy()
y = win_df["label"].to_numpy()
groups = win_df["subject_id"].to_numpy()

gkf = GroupKFold(n_splits=5)
for name, model in [("Logistic Regression", LogisticRegression(max_iter=1000)),
                     ("Random Forest", RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0))]:
    accs = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        m = model.__class__(**model.get_params())
        m.fit(X[train_idx], y[train_idx])
        pred = m.predict(X[test_idx])
        accs.append(accuracy_score(y[test_idx], pred))
    print(f"{name}: mean accuracy={np.mean(accs):.3f}, worst fold={np.min(accs):.3f}")

majority_baseline = max((y == 0).mean(), (y == 1).mean())
print(f"majority class baseline (always guess the more common label): {majority_baseline:.3f}")

rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0)
rf.fit(X, y)
print()
print("what the random forest classifier leans on:")
for f, i in sorted(zip(phase_feature_cols, rf.feature_importances_), key=lambda t: -t[1]):
    print(f"  {f}: {i:.3f}")

windows built: 8798, from 31 recordings, 21 subjects
concentric: 4867, eccentric: 3931


Logistic Regression: mean accuracy=0.788, worst fold=0.732


Random Forest: mean accuracy=0.799, worst fold=0.750
majority class baseline (always guess the more common label): 0.553



what the random forest classifier leans on:
  slope_accel: 0.766
  mean_accel: 0.074
  mean_gyro: 0.063
  abs_mean_accel: 0.060
  std_accel: 0.024
  std_gyro: 0.013


The classifier reaches about 0.79 to 0.80 mean accuracy, well above the
0.55 a model gets by always guessing the more common label, using windows of
only 0.3 seconds and no information about where in the rep cycle that window
sits. The feature it relies on almost entirely is the slope of the smoothed
signal within the window, whether it is rising or falling. That matches the
physical picture directly, the concentric phase accelerates the bar upward,
the eccentric phase lets it come back down, and a short window is often
enough to tell which is happening.

This result should be read carefully. It is real, it beats the baseline by a
wide margin, and it generalizes to subjects the model never trained on. But
the accuracy number measures agreement with the geometric rule used to label
the data, not agreement with an independently verified ground truth, because
no such ground truth exists in this dataset. What this stage actually shows
is that phase information is recoverable from a short local window, which
means a live system would not need a full recording's worth of data before it
could tell a user which half of the movement they are in.

# Conclusion

Three tasks were asked for in this notebook, and three trained, cross
validated results now exist. Exercise recognition uses a logistic regression
model and reaches about 0.97 mean accuracy across subject grouped folds,
worst fold about 0.92, and three different model types land in the same
range, so the result comes from the features and the problem being easy, not
from any one model being unusually good. Rep recognition splits by exercise,
Chest Press now uses a random forest regression model that reaches a mean
absolute error of about 1.2 reps, an improvement over the 1.9 reps of the
signal processing method and a fix for that method's worst failure case,
while Lateral Raise keeps its original signal processing method because
retraining it did not improve on 1.63 reps of error. Phase recognition uses a
random forest classifier that separates the concentric and eccentric halves
of a Chest Press rep at about 0.80 accuracy from a 0.55 baseline, with the
caveat that its labels come from a physically grounded rule rather than an
independent annotation.

Shoulder Press stays out of this notebook by decision. Recognition,
counting, and phase splitting have only been shown to work on the easiest
possible pair of exercises, and that should not be read as more general than
it is.

What is not covered here has not changed. The weak point ratio signal from
earlier in this project is a real, measurable pattern, not yet a trained
model and not yet validated against any ground truth, since none exists in
this dataset. Failure tracking still needs a decision about what proxy to use
for speed, since double integration to real velocity has already been shown,
twice, to drift too much to trust. Session tracking needs the `multionly`
version of the dataset, which this project has not touched.